In [16]:
%run Utils.ipynb

In [2]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from pandas.plotting import autocorrelation_plot
%matplotlib inline
from matplotlib import pyplot as plt

In [3]:
dict_files = get_files(n=None)


In [4]:
# dict_files.keys()
estacao = "A401"

In [5]:
print(dict_files[estacao][0])

REGIAO:;NE
UF:;BA
ESTACAO:;SALVADOR
CODIGO (WMO):;A401
LATITUDE:;-13,005515
LONGITUDE:;-38,50576
ALTITUDE:;47,56
DATA DE FUNDACAO:;13/05/00



### Pre processamento

In [6]:
def pre_processing1(path):
    df = load(path)
    df = renameDatasetColumns(df)
    df = transform_datetime(df)
    df = create_datetime_feature(df)
    df = create_split_date_features(df)
    df = removeNulls(df, "radiacao")
    df = change_types(df)
    return treat_columns(df)

def pre_processing2(path, cidade):
    df = load(path)
    df = renameDatasetColumns(df)
    df = transform_datetime(df)
    df = create_datetime_feature(df)
    df = create_split_date_features(df)
    df = change_types(df)
    df["cidade"] = cidade
    return df


In [7]:
df = pre_processing1(dict_files["A401"][1])

In [34]:
path = dict_files[estacao][1]
# pre_processing(load(dict_files["A401"][1]))
df_A401 = pre_processing(load(dict_files["A401"][1]))
# df_A413 = pre_processing(dict_files["A413"][1])[["hora", "radiacao"]]
# df_A440 = pre_processing(dict_files["A440"][1])[["hora", "radiacao"]]
df_A801 = pre_processing(load(dict_files["A801"][1]))
# df_A840 = pre_processing(dict_files["A840"][1])[["hora", "radiacao"]]
# df_A893 = pre_processing(dict_files["A893"][1])[["hora", "radiacao"]]

In [56]:
df_A401_verao = df_A401[(df_A401["data"] >= "2023-12-01") & (df_A401["data"] <= "2023-12-20")]
df_A801_verao = df_A801[(df_A801["data"] >= "2023-12-01") & (df_A801["data"] <= "2023-12-20")]

In [63]:
df_A401_verao_grouped = df_A401_verao.groupby(['mes'])\
['precipitacao'].agg(['mean']).reset_index()
df_A401_verao_grouped["cidade"] = "Salvador"

In [64]:
df_A801_verao_grouped = df_A801_verao.groupby(['mes'])\
['precipitacao'].agg(['mean']).reset_index()
df_A801_verao_grouped["cidade"] = "Rio Grande do Sul"

In [65]:
pd.concat([df_A401_verao_grouped, df_A801_verao_grouped], axis=0)

,mes,mean,cidade
0,12,0.206923,Salvador
0,12,0.070990,Rio Grande do Sul


### Algumas estatisticas

In [ ]:
df_A401 = pre_processing2(dict_files["A401"][1], "Salvador")[["radiacao", "cidade"]]
df_A413 = pre_processing2(dict_files["A413"][1], "Feira de Santana")[["radiacao", "cidade"]]
df_A440 = pre_processing2(dict_files["A440"][1], "Jacobina")[["radiacao", "cidade"]]
df_A801 = pre_processing2(dict_files["A801"][1], "Porto Alegre")[["radiacao", "cidade"]]
df_A840 = pre_processing2(dict_files["A840"][1], "Bento Gonçalves")[["radiacao", "cidade"]]
df_A893 = pre_processing2(dict_files["A893"][1], "Encruzilhada do Sul")[["radiacao", "cidade"]]

df_final = pd.concat([df_A401, df_A413, df_A440, df_A801, df_A840, df_A893], axis=0)

df_final.groupby(['cidade'])\
.agg(['mean','std'])

In [ ]:
df_A401 = pre_processing2(dict_files["A401"][1], "Salvador")[["radiacao", "cidade"]]
df_A413 = pre_processing2(dict_files["A413"][1], "Feira de Santana")[["radiacao", "cidade"]]
df_A440 = pre_processing2(dict_files["A440"][1], "Jacobina")[["radiacao", "cidade"]]
df_A801 = pre_processing2(dict_files["A801"][1], "Porto Alegre")[["radiacao", "cidade"]]
df_A840 = pre_processing2(dict_files["A840"][1], "Bento Gonçalves")[["radiacao", "cidade"]]
df_A893 = pre_processing2(dict_files["A893"][1], "Encruzilhada do Sul")[["radiacao", "cidade"]]

df_final = pd.concat([df_A401, df_A413, df_A440, df_A801, df_A840, df_A893], axis=0)

df_final_null = df_final.radiacao.isnull().groupby(df_final['cidade']).sum().astype(int).reset_index(name='null')
df_final_not_null = df_final.radiacao.notnull().groupby(df_final['cidade']).sum().astype(int).reset_index(name='not_null')

df_final_2 = df_final_null.merge(df_final_not_null, left_on='cidade', right_on='cidade')

display(df_final_2)

In [ ]:
set_plot_size(11, 4)

df_hour_A401 = df_A401[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()


df_hour_A413 = df_A413[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()
df_hour_A413["cidade"] = "Feira de Santana"

df_hour_A440 = df_A440[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()
df_hour_A440["cidade"] = "Jacobina"

df_hour_A801 = df_A801[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()
df_hour_A801["cidade"] = "Porto Alegre"

df_hour_A840 = df_A840[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()
df_hour_A840["cidade"] = "Bento Gonçalves"

df_hour_A893 = df_A893[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()
df_hour_A893["cidade"] = "Encruzilhada do Sul"


df_final = pd.concat([df_hour_A401, df_hour_A413, df_hour_A440, df_hour_A801, df_hour_A840, df_hour_A893], axis=0)

# ax1 = plt.subplot()
# ax2 = ax1.twinx()
# ax3 = ax2.twinx()
# ax4 = ax3.twinx()
# ax5 = ax4.twinx()
# ax6 = ax5.twinx()

# df_final.head()
sns.lineplot(data=df_final, x='hora', y='mean', hue="cidade")
# sns.lineplot(data=df_hour_A413, x='hora', y='mean', ax=ax2, color='b', hue="cidade")
# sns.lineplot(data=df_hour_A440, x='hora', y='mean', ax=ax3, color='g', hue="cidade")
# sns.lineplot(data=df_hour_A801, x='hora', y='mean', ax=ax4, color='r', hue="cidade")
# sns.lineplot(data=df_hour_A840, x='hora', y='mean', ax=ax5, color='p', hue="cidade")
# sns.lineplot(data=df_hour_A893, x='hora', y='mean', ax=ax6, color='assaas', hue="cidade")

In [ ]:
set_plot_size(11, 4)

df_hour_mean = df[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['mean']).reset_index()

df_hour_std = df[["hora", "radiacao"]].groupby(['hora'])\
['radiacao'].agg(['std']).reset_index()

ax1 = plt.subplot()
ax2 = ax1.twinx()
sns.lineplot(data=df_hour_mean, x='hora', y='mean', ax=ax1)
sns.lineplot(data=df_hour_std, x='hora', y='std', color='r', ax=ax2)

# sns.lineplot(data=df_hour, x=df_hour["hora"], )

#### Usar esse grafico pra evidenciar a estacionariedade

In [ ]:
set_plot_size(11, 2)
df_test = filter_between(df, "data", "2023-01-01", "2023-01-07")
df_test["mean"] = df_test["radiacao"].mean()

df_test[["mean", "radiacao"]].plot.line()
# df_test[["data", "radiacao"]].groupby(['data'])\
# ['radiacao'].agg(['mean','std']).plot.line()
# sns.lineplot(data=df_test, x=df_test["data_hora"], y=df_test["radiacao"])

#### proximo passo: plotar a corerlação entre a serie e ela defasada

In [ ]:
df_test = df[["data_hora", "radiacao"]]
df_test["radiacao_shift"] = df_test["radiacao"].shift(periods=8)

In [ ]:
# df_test.head(50)

In [ ]:
set_plot_size(11, 5)
sns.scatterplot(data=df_test, x=df_test["radiacao"], y=df_test["radiacao_shift"])

In [ ]:
df_test = df[["data_hora", "data", "radiacao", "dia"]]
df_test = filter_between(df_test, "data", "2023-01-01", "2023-01-28")


def teste_corr():
    pass

group_1 = df_test[(df_test["dia"] >=1) & (df_test["dia"] <= 7)]\
.reset_index()\
.drop("index", axis=1)\
.reset_index()\
[["index", "radiacao"]]\
.rename(columns={"radiacao": "radiacao_1"})

group_2 = df_test[(df_test["dia"] >=8) & (df_test["dia"] <= 14)]\
.reset_index()\
.drop("index", axis=1)\
.reset_index()\
[["index", "radiacao"]]\
.rename(columns={"radiacao": "radiacao_2"})


group_3 = df_test[(df_test["dia"] >=15) & (df_test["dia"] <= 21)]\
.reset_index()\
.drop("index", axis=1)\
.reset_index()\
[["index", "radiacao"]]\
.rename(columns={"radiacao":"radiacao_3"})

group_4 = df_test[(df_test["dia"] >=22) & (df_test["dia"] <= 28)]\
.reset_index()\
.drop("index", axis=1)\
.reset_index()\
[["index", "radiacao"]]\
.rename(columns={"radiacao":"radiacao_4"})

df_final = group_1\
.merge(group_2, left_on='index', right_on='index')\
.merge(group_3, left_on='index', right_on='index')\
.merge(group_4, left_on='index', right_on='index')

df_final.head()

In [ ]:
set_plot_size(11, 3)
df_final.plot.line()

In [ ]:
sns.scatterplot(data=df_final, x=df_final["radiacao_1"], y=df_final["radiacao_2"])

In [ ]:
sns.scatterplot(data=df_final, x=df_final["radiacao_1"], y=df_final["radiacao_3"])

In [ ]:
sns.scatterplot(data=df_final, x=df_final["radiacao_1"], y=df_final["radiacao_4"])

In [ ]:
corr = df_final.drop("index", axis=1).corr()

In [ ]:
corr

In [ ]:
sns.heatmap(corr)

In [ ]:
corr = df_final.drop("index", axis=1).corr()